# LoRA, QLoRA, DoRA & Beyond — Week 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain *why* parameter-efficient fine-tuning (PEFT) exists and the memory math behind it
2. Describe the LoRA decomposition `W = W₀ + BA` and what rank `r` controls
3. Compare four LoRA variants (vanilla, RSLoRA, DoRA, PiSSA) and choose between them
4. Use `lora_setup.build_lora_config` and `count_trainable_params` to explore rank/alpha trade-offs

## Time Estimate
~30 minutes (15 min reading + 15 min experiments)

In [2]:
import sys, importlib
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

import src.lora_setup as lora_setup
importlib.reload(lora_setup)

from src.lora_setup import build_lora_config, count_trainable_params, save_lora_scoreboard
from src.config import FINETUNE_BACKEND, BASE_MODEL_HF
from src.cost_tracker import CostTracker
from src.utils import append_to_reflection

tracker = CostTracker()
print(f"Backend: {FINETUNE_BACKEND}")
print(f"Base model: {BASE_MODEL_HF}")

Backend: hf
Base model: Qwen/Qwen2.5-0.5B-Instruct


---
## Part 1: Why Parameter-Efficient Fine-Tuning?

Full fine-tuning updates every parameter in the model. For a 7B-parameter model stored in fp16 that's already **28 GB** just for weights — before optimizer states (Adam needs 2× params = 56 GB more) and activations. A single A100-80GB can't hold it.

**The key insight behind LoRA** (Hu et al., 2021): when you adapt a model to a new task, the *update* to each weight matrix is empirically low-rank. That is, you don't need to change every element; the change can be approximated by the product of two thin matrices:

```
W_new = W₀ + ΔW
      = W₀ + B · A
```

where:
- `W₀` is the **frozen** pre-trained weight (shape `d × d`)
- `B` has shape `d × r`  (down-projection)
- `A` has shape `r × d`  (up-projection)
- `r` is the **rank**, typically 4–64 (versus d ≈ 4096 for a 7B model)

Only `B` and `A` are trained. Parameter count goes from `d²` → `2·d·r`.

For Qwen-2.5-0.5B (494M total params), with r=16 targeting 7 projection layers:
- **~0.8% of parameters** are trainable
- Adapter fits in a few hundred MB instead of gigabytes
- Base weights stay frozen → no catastrophic forgetting of general knowledge

In [3]:
# Experiment 1: Build a vanilla LoRA config and inspect it
cfg_r16 = build_lora_config(rank=16, alpha=32, variant='vanilla')
print()
print("LoraConfig object:", cfg_r16)

# Expected trainable parameter percentage for Qwen-0.5B with r=16:
TOTAL_PARAMS = 494_032_896  # Qwen2.5-0.5B-Instruct
# 7 target modules × 2 projection layers each × (hidden_dim × rank + rank × hidden_dim)
# Qwen-0.5B hidden_dim=896, num_layers=24, 7 proj per layer
# Each LoRA pair: 2 * hidden_dim * rank
hidden_dim = 896
num_layers = 24
n_modules = 7
rank = 16
lora_params = num_layers * n_modules * 2 * hidden_dim * rank
pct = 100.0 * lora_params / TOTAL_PARAMS
print(f"\nEstimated trainable params (r=16, 7 modules, 24 layers): {lora_params:,}")
print(f"Trainable %: {pct:.2f}%")
print(f"(This is ~0.8% — tiny compared to full fine-tune)")

[lora_setup] LoraConfig built:
  variant         : vanilla
  rank (r)        : 16
  alpha           : 32
  lora_dropout    : 0.05
  target_modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  task_type       : CAUSAL_LM
  use_rslora      : False
  use_dora        : False
  init_lora_weights: True

LoraConfig object: LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'down_proj', 'q_proj', 'up_proj', 'o_proj', 'v_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_con

---
## Part 2: Rank and Alpha Exploration

The rank `r` is the primary hyperparameter in LoRA. Alpha (`α`) controls the scaling of the LoRA output:
```
output = W₀·x + (α/r) · B·A·x
```
A common heuristic is `α = 2r` (so the scaling factor is 2.0).

| Rank | Trainable % (Qwen-0.5B) | Adapter Memory | Quality Trade-off |
|------|------------------------|----------------|-------------------|
| 4    | ~0.2%                  | ~40 MB         | Good for style/tone; may miss complex patterns |
| 8    | ~0.4%                  | ~80 MB         | Solid general-purpose; recommended for most tasks |
| 16   | ~0.8%                  | ~160 MB        | Better at knowledge-heavy tasks; our default |
| 32   | ~1.6%                  | ~320 MB        | Diminishing returns; good for hard reasoning tasks |
| 64   | ~3.2%                  | ~640 MB        | Rarely better than r=32; risk of overfitting small datasets |

In [5]:
# Experiment 2: Build configs for multiple ranks, compare parameter counts
ranks = [4, 8, 16, 32]
results = []

for r in ranks:
    lora_params_r = num_layers * n_modules * 2 * hidden_dim * r
    pct_r = 100.0 * lora_params_r / TOTAL_PARAMS
    adapter_mb = (lora_params_r * 2) / (1024 ** 2)  # fp16 = 2 bytes
    results.append({
        'rank': r,
        'alpha': r * 2,
        'trainable_params': lora_params_r,
        'trainable_pct': round(pct_r, 3),
        'adapter_mb': round(adapter_mb, 1),
    })

print(f"{'Rank':>6} {'Alpha':>6} {'Trainable Params':>18} {'Trainable %':>12} {'Adapter (MB)':>14}")
print("-" * 62)
for r in results:
    print(f"{r['rank']:>6} {r['alpha']:>6} {r['trainable_params']:>18,} {r['trainable_pct']:>11.3f}% {r['adapter_mb']:>13.1f}")

  Rank  Alpha   Trainable Params  Trainable %   Adapter (MB)
--------------------------------------------------------------
     4      8          1,204,224       0.244%           2.3
     8     16          2,408,448       0.488%           4.6
    16     32          4,816,896       0.975%           9.2
    32     64          9,633,792       1.950%          18.4


### TODO 1: Absolute Parameter Count

If rank=16 gives approximately **0.8% trainable params**, how many *absolute* parameters is that for Qwen-0.5B (494M total)?

Calculate this below and then answer:
- How does that number compare to a full fine-tune?
- Why does having fewer trainable parameters matter for memory during training (hint: think about optimizer states)?

In [6]:
# TODO 1: Calculate absolute trainable params for r=16
# YOUR CODE HERE

total,params = 494,032,896
pct,trainable = 0.8 / 100  # ~0.8%

# Q1: How many absolute params?
# absolute,trainable = 494,032,896 * 0.008 = 3,952,263

# Q2: Compare to full fine-tune
# full,params = total,params = 494,032,896
# ratio = 494,032,896 / 3,952,263 = 125

# Q3: Memory impact — Adam optimizer stores (param + grad + m + v) = 4 × params × 4 bytes (fp32)
# lora,optimizer,bytes = 4 * 3,952,263 * 4 =  63,236,208 = 63 MB
# full,optimizer,bytes = 4 * 494,032,896 * 4 =  7,904,526,336  =7.9 GB

print("TODO 1: Fill in the calculations above.")
print("Remember: Adam stores 4 tensors per parameter (param, grad, momentum, variance)")
print("At fp32 that's 4 × 4 bytes = 16 bytes per parameter.")

TODO 1: Fill in the calculations above.
Remember: Adam stores 4 tensors per parameter (param, grad, momentum, variance)
At fp32 that's 4 × 4 bytes = 16 bytes per parameter.


In [7]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """
# Q1: How many absolute params?
# absolute,trainable = 494,032,896 * 0.008 = 3,952,263

# Q2: Compare to full fine-tune
# full,params = total,params = 494,032,896
# ratio = 494,032,896 / 3,952,263 = 125

# Q3: Memory impact — Adam optimizer stores (param + grad + m + v) = 4 × params × 4 bytes (fp32)
# lora,optimizer,bytes = 4 * 3,952,263 * 4 =  63,236,208 = 63 MB
# full,optimizer,bytes = 4 * 494,032,896 * 4 =  7,904,526,336  =7.9 GB

This matters on a 16GB Mac because Adam optimizer memory scales with the number of trainable parameters. The optimizer-related memory is about 60 MiB for LoRA versus about 7.36 GiB for full fine-tuning. Saving that much memory makes training much more feasible on limited RAM/VRAM.
"""
print(todo1_reflection)



# Q1: How many absolute params?
# absolute,trainable = 494,032,896 * 0.008 = 3,952,263

# Q2: Compare to full fine-tune
# full,params = total,params = 494,032,896
# ratio = 494,032,896 / 3,952,263 = 125

# Q3: Memory impact — Adam optimizer stores (param + grad + m + v) = 4 × params × 4 bytes (fp32)
# lora,optimizer,bytes = 4 * 3,952,263 * 4 =  63,236,208 = 63 MB
# full,optimizer,bytes = 4 * 494,032,896 * 4 =  7,904,526,336  =7.9 GB

This matters on a 16GB Mac because Adam optimizer memory scales with the number of trainable parameters. The optimizer-related memory is about 60 MiB for LoRA versus about 7.36 GiB for full fine-tuning. Saving that much memory makes training much more feasible on limited RAM/VRAM.



---
## Part 3: LoRA Variants — RSLoRA, DoRA, PiSSA

The original LoRA paper scales the adapter output by `α/r`. At high ranks this can cause gradient instability. Several improved variants address this:

### RSLoRA (Rank-Stabilized LoRA)
- Changes the scaling from `α/r` to `α/√r`
- Result: gradients remain stable as rank increases — you can use r=64 without divergence
- Minimal overhead; a drop-in replacement for vanilla LoRA

### DoRA (Weight-Decomposed LoRA)
- Decomposes each weight matrix into **magnitude** and **direction** components
- The direction is updated with a standard LoRA adapter; magnitude is a learnable scalar per output feature
- Mimics full fine-tuning more closely; especially effective for instruction following and reasoning
- ~2-3% more parameters than vanilla LoRA due to magnitude vectors

### PiSSA (Principal Singular Values and Singular Vectors Adaptation)
- Initializes A and B from the **SVD** of W₀ (taking the top-r singular components)
- The residual (small singular values) remains frozen as the effective W₀
- Converges faster than random initialization (LoRA A starts at zeros/random, B at zero)
- Good when you have limited steps (e.g., your compute budget is tight)

In [8]:
# Experiment 3: Build all four variants and inspect their key parameters
variants = ['vanilla', 'rslora', 'dora', 'pissa']
variant_configs = {}

for v in variants:
    print(f"\n{'='*50}")
    print(f"Building config: variant={v}")
    cfg = build_lora_config(rank=16, alpha=32, variant=v)
    variant_configs[v] = cfg

print("\n" + "="*60)
print("VARIANT SUMMARY TABLE")
print("="*60)
print(f"{'Variant':<12} {'use_rslora':<12} {'use_dora':<10} {'init_weights':<16} {'rank':<6} {'alpha':<6}")
print("-" * 62)
for v, cfg in variant_configs.items():
    use_rslora = getattr(cfg, 'use_rslora', False)
    use_dora   = getattr(cfg, 'use_dora', False)
    init_w     = str(getattr(cfg, 'init_lora_weights', True))
    print(f"{v:<12} {str(use_rslora):<12} {str(use_dora):<10} {init_w:<16} {cfg.r:<6} {cfg.lora_alpha:<6}")


Building config: variant=vanilla
[lora_setup] LoraConfig built:
  variant         : vanilla
  rank (r)        : 16
  alpha           : 32
  lora_dropout    : 0.05
  target_modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  task_type       : CAUSAL_LM
  use_rslora      : False
  use_dora        : False
  init_lora_weights: True

Building config: variant=rslora
[lora_setup] LoraConfig built:
  variant         : rslora
  rank (r)        : 16
  alpha           : 32
  lora_dropout    : 0.05
  target_modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  task_type       : CAUSAL_LM
  use_rslora      : True

Building config: variant=dora
[lora_setup] LoraConfig built:
  variant         : dora
  rank (r)        : 16
  alpha           : 32
  lora_dropout    : 0.05
  target_modules  : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  task_type       : CAUSAL_LM
  use_dora        : True

Bui

In [9]:
# Build scoreboard and save it
scoreboard = {}
for v in variants:
    # Estimate trainable params analytically (avoids loading model weights)
    lora_params_v = num_layers * n_modules * 2 * hidden_dim * 16
    # DoRA adds one magnitude scalar per output neuron per module
    if v == 'dora':
        lora_params_v += num_layers * n_modules * hidden_dim  # magnitude vectors
    pct_v = round(100.0 * lora_params_v / TOTAL_PARAMS, 4)
    scoreboard[v] = {
        'trainable_params': lora_params_v,
        'total_params': TOTAL_PARAMS,
        'trainable_pct': pct_v,
        'use_rslora': getattr(variant_configs[v], 'use_rslora', False),
        'use_dora': getattr(variant_configs[v], 'use_dora', False),
        'init_lora_weights': str(getattr(variant_configs[v], 'init_lora_weights', True)),
        'rank': 16,
        'alpha': 32,
    }

save_lora_scoreboard(scoreboard, path='../outputs/lora_variants_scoreboard.json')

print("\nScoreboard saved. Summary:")
for v, info in scoreboard.items():
    print(f"  {v:<10}: {info['trainable_params']:>12,} trainable params ({info['trainable_pct']:.4f}%)")

[lora_setup] Scoreboard saved to ../outputs/lora_variants_scoreboard.json (4 variants)

Scoreboard saved. Summary:
  vanilla   :    4,816,896 trainable params (0.9750%)
  rslora    :    4,816,896 trainable params (0.9750%)
  dora      :    4,967,424 trainable params (1.0055%)
  pissa     :    4,816,896 trainable params (0.9750%)


### TODO 2: Choosing a Production Variant

You're about to fine-tune `Qwen2.5-0.5B-Instruct` for a resume Q&A assistant on a Mac M3 with 16GB unified memory. You have ~2 hours of compute budget.

**In the cell below, write 3 sentences justifying your variant choice.** Specifically:
- Compare RSLoRA vs DoRA: when is the overhead of DoRA worth it?
- Would PiSSA's faster convergence matter given your 2-hour budget?
- What rank would you pair with your chosen variant?

In [11]:
# TODO 2: Write your justification as a string
todo2_response = """
I would choose RSLoRA because Qwen2.5-0.5B is small enough that a lightweight, stable LoRA variant is more practical than adding extra overhead or complexity.

RSLoRA vs DoRA: RSLoRA is the better fit for this constraint because it has almost no overhead, while DoRA's extra per-step cost is worth it mainly when maximum instruction-following quality matters more than training speed.

PiSSA convergence consideration: PiSSA's faster convergence could help within a 2-hour budget, but for this small resume Q&A assistant I would prioritize RSLoRA's simplicity, stability, and low overhead.

Rank choice: I would use r=16 with alpha=32 because it gives a strong quality/memory tradeoff without pushing memory or training time too high on a 16GB Mac.
"""

print(todo2_response)


I would choose RSLoRA because Qwen2.5-0.5B is small enough that a lightweight, stable LoRA variant is more practical than adding extra overhead or complexity.

RSLoRA vs DoRA: RSLoRA is the better fit for this constraint because it has almost no overhead, while DoRA's extra per-step cost is worth it mainly when maximum instruction-following quality matters more than training speed.

PiSSA convergence consideration: PiSSA's faster convergence could help within a 2-hour budget, but for this small resume Q&A assistant I would prioritize RSLoRA's simplicity, stability, and low overhead.

Rank choice: I would use r=16 with alpha=32 because it gives a strong quality/memory tradeoff without pushing memory or training time too high on a 16GB Mac.



In [13]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """
I would choose RSLoRA because Qwen2.5-0.5B is small enough that a lightweight, stable LoRA variant is more practical than adding extra overhead or complexity. 
1. RSLoRA is the better fit than DoRA under these constraints because it has almost no overhead, while DoRA's extra per-step cost is worth it mainly when maximum instruction-following quality matters more than training speed. 
2. PiSSA's faster convergence could help within a 2-hour budget, but for this small resume Q&A assistant I would prioritize RSLoRA.
3. I would pair it with r=16 and alpha=32 because it gives a strong quality/memory tradeoff on a 16GB Mac.
"""
print(todo2_reflection)



I would choose RSLoRA because Qwen2.5-0.5B is small enough that a lightweight, stable LoRA variant is more practical than adding extra overhead or complexity. 
1. RSLoRA is the better fit than DoRA under these constraints because it has almost no overhead, while DoRA's extra per-step cost is worth it mainly when maximum instruction-following quality matters more than training speed. 
2. PiSSA's faster convergence could help within a 2-hour budget, but for this small resume Q&A assistant I would prioritize RSLoRA.
3. I would pair it with r=16 and alpha=32 because it gives a strong quality/memory tradeoff on a 16GB Mac.



---
## Summary

In this notebook you:
- Derived why LoRA reduces trainable params by ~99% for Qwen-0.5B
- Compared ranks 4/8/16/32 on memory and parameter counts
- Built all four LoRA variants (vanilla, RSLoRA, DoRA, PiSSA) and saved a scoreboard

**Key takeaways:**
- For most tasks, **rank=8 or rank=16 with alpha=2r** is a solid default
- **RSLoRA** is a free upgrade over vanilla — just set `use_rslora=True`
- **DoRA** is worth the overhead when you need strong instruction-following quality
- **PiSSA** shines with very limited training steps (fast convergence)

Next: `05_sft_with_trl.ipynb` — actually training with these configs!

In [14]:
# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1: Absolute Parameter Count\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2: Choosing a Production Variant\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="04",
    section_title="LoRA, QLoRA, DoRA & Beyond",
    reflection_content=section_text,
    output_dir="../outputs",
)
print("Reflection auto-saved to outputs/homework_reflection.md")
tracker.report()


Reflection auto-saved to outputs/homework_reflection.md
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

